## 1. Import and preprocess

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from category_encoders import CountEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import statsmodels
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
df = pd.read_csv(filepath_or_buffer='data/training.csv', index_col=0, parse_dates=['PurchDate'])
df['PRIMEUNIT'] = df['PRIMEUNIT'].fillna('MISSING')
df['AUCGUART'] = df['AUCGUART'].fillna('MISSING')
df.isna().sum()

IsBadBuy                                0
PurchDate                               0
Auction                                 0
VehYear                                 0
VehicleAge                              0
Make                                    0
Model                                   0
Trim                                 2360
SubModel                                8
Color                                   8
Transmission                            9
WheelTypeID                          3169
WheelType                            3174
VehOdo                                  0
Nationality                             5
Size                                    5
TopThreeAmericanName                    5
MMRAcquisitionAuctionAveragePrice      18
MMRAcquisitionAuctionCleanPrice        18
MMRAcquisitionRetailAveragePrice       18
MMRAcquisitonRetailCleanPrice          18
MMRCurrentAuctionAveragePrice         315
MMRCurrentAuctionCleanPrice           315
MMRCurrentRetailAveragePrice      

In [3]:
y = df['IsBadBuy']
X = df[df.columns[1:]]
X

,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
RefId,,,,,,,,,,,,,,,,,,,,,
1,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,...,11597.0,12409.0,MISSING,MISSING,21973,33619,FL,7100.0,0,1113
2,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,...,11374.0,12791.0,MISSING,MISSING,19638,33619,FL,7600.0,0,1053
3,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,...,7146.0,8702.0,MISSING,MISSING,19638,33619,FL,4900.0,0,1389
4,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,...,4375.0,5518.0,MISSING,MISSING,19638,33619,FL,4100.0,0,630
5,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,...,6739.0,7911.0,MISSING,MISSING,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73010,2009-12-02,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,BLACK,AUTO,...,4836.0,5937.0,MISSING,MISSING,18111,30212,GA,4200.0,0,993
73011,2009-12-02,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,SILVER,AUTO,...,10151.0,11652.0,MISSING,MISSING,18881,30212,GA,6200.0,0,1038
73012,2009-12-02,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,SILVER,AUTO,...,11831.0,14402.0,MISSING,MISSING,18111,30212,GA,8200.0,0,1893


In [4]:
def my_train_valid_test_date_split(X, y, validation_date, test_date):
    train_mask = X['PurchDate'] < validation_date
    valid_mask = (X['PurchDate'] >= validation_date) & (X['PurchDate'] <= test_date)
    test_mask = X['PurchDate'] > test_date

    X_train = X[train_mask]
    X_valid = X[valid_mask]
    X_test = X[test_mask]

    y_train = y[train_mask]
    y_valid = y[valid_mask]
    y_test = y[test_mask]

    X_train = X_train.drop(labels=['PurchDate'], axis=1)
    X_valid = X_valid.drop(labels=['PurchDate'], axis=1)
    X_test = X_test.drop(labels=['PurchDate'], axis=1)

    return X_train, X_valid, X_test, y_train, y_valid, y_test

In [5]:
first_date = df['PurchDate'].quantile(0.34)
second_date = df['PurchDate'].quantile(0.67)
print(first_date)
print(second_date)

2009-09-17 00:00:00
2010-05-18 00:00:00


In [6]:
X_train, X_valid, X_test, y_train, y_valid, y_test = my_train_valid_test_date_split(X, y, first_date, second_date)
print(len(X_train), len(X_valid), len(X_test))

24695 24372 23916


In [7]:
fill_values = {
    'WheelType':    X_train['WheelType'].mode()[0],
    'WheelTypeID':  X_train['WheelTypeID'].mode()[0],
    'Trim':         X_train['Trim'].mode()[0],
    'SubModel':     X_train['SubModel'].mode()[0],
    'Color':        X_train['Color'].mode()[0],
    'Transmission': X_train['Transmission'].mode()[0],
    'Nationality':  X_train['Nationality'].mode()[0],
    'Size':         X_train['Size'].mode()[0],
    'TopThreeAmericanName': X_train['TopThreeAmericanName'].mode()[0],
    'MMRAcquisitionAuctionAveragePrice': X_train['MMRAcquisitionAuctionAveragePrice'].median(),
    'MMRAcquisitionAuctionCleanPrice':   X_train['MMRAcquisitionAuctionCleanPrice'].median(),
    'MMRAcquisitionRetailAveragePrice':  X_train['MMRAcquisitionRetailAveragePrice'].median(),
    'MMRAcquisitonRetailCleanPrice':     X_train['MMRAcquisitonRetailCleanPrice'].median(),
    'MMRCurrentAuctionAveragePrice':     X_train['MMRCurrentAuctionAveragePrice'].median(),
    'MMRCurrentAuctionCleanPrice':       X_train['MMRCurrentAuctionCleanPrice'].median(),
    'MMRCurrentRetailAveragePrice':      X_train['MMRCurrentRetailAveragePrice'].median(),
    'MMRCurrentRetailCleanPrice':        X_train['MMRCurrentRetailCleanPrice'].median(),
}

X_train.fillna(fill_values, inplace=True)
X_valid.fillna(fill_values, inplace=True)
X_test.fillna(fill_values, inplace=True)

In [8]:
categorical_cols = [
    'Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color',
    'Transmission', 'WheelType', 'Nationality', 'Size',
    'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST',
    'WheelTypeID', 'BYRNO', 'VNZIP1'
]

In [9]:
encoder = CountEncoder(cols=categorical_cols)
X_train = encoder.fit_transform(X_train)
X_valid = encoder.transform(X_valid)
X_test = encoder.transform(X_test)
X_test

,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
RefId,,,,,,,,,,,,,,,,,,,,,
266,5076,2006,4,5743.0,0.0,3039.0,142.0,109,24008,10788.0,...,12427.0,13850.0,24693.0,24693.0,545.0,458.0,3452.0,6800.0,0,1630
267,5076,2004,6,5743.0,0.0,183.0,493.0,1921,24008,13666.0,...,9497.0,11161.0,24693.0,24693.0,913.0,458.0,3452.0,8600.0,0,1020
268,5076,2006,4,2832.0,3.0,5359.0,4991.0,2048,24008,10788.0,...,8667.0,9390.0,24693.0,24693.0,913.0,458.0,3452.0,6150.0,0,1215
269,5076,2007,3,656.0,9.0,210.0,179.0,729,24008,10788.0,...,9792.0,10587.0,24693.0,24693.0,913.0,458.0,3452.0,5800.0,0,728
270,5076,2002,8,287.0,1.0,233.0,106.0,2048,24008,13666.0,...,8950.0,10403.0,24693.0,24693.0,545.0,458.0,3452.0,7600.0,0,1543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72415,5076,2004,6,656.0,0.0,47.0,47.0,2504,24008,241.0,...,6364.0,7176.0,24693.0,24693.0,0.0,409.0,4615.0,4105.0,0,2891
72416,5076,2003,7,4852.0,2.0,434.0,12.0,2048,24008,13666.0,...,7853.0,8917.0,0.0,0.0,0.0,409.0,4615.0,4490.0,0,1930
72417,5076,2002,8,4323.0,2.0,3484.0,1448.0,3486,24008,13666.0,...,5411.0,6523.0,0.0,0.0,0.0,409.0,4615.0,3690.0,0,1455


In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

### 2. Create a Python class for Decision Tree Classifier and Decision Tree Regressor (MSE loss).



In [11]:
def gini_imp(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1 - np.sum(p ** 2)

In [12]:
class NodeClassifier:
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.feature = None
        self.left = None
        self.right = None
        self.threshold = None
        self.value = None
        self.classes, self.counts = np.unique(self.y, return_counts=True)

    def gini(self):
        p = self.counts / len(self.y)
        return 1 - np.sum(p ** 2)

    def split(self):
        best_score = 10000000
        best_feature = None
        best_threshold = None
        for j in range(self.X.shape[1]):
            for t in np.unique(self.X[:, j]):
                N = len(self.y)
                left_mask = self.X[:, j] <= t
                yl = self.y[left_mask]
                yr = self.y[~left_mask]
                if len(yl) == 0 or len(yr) == 0: continue
                score = (len(yl) / N) * gini_imp(yl) + (len(yr) / N) * gini_imp(yr)
                #score = (len(left_node.y) / N) * left_node.gini() + (len(right_node.y) / N) * right_node.gini()
                if score < best_score:
                    best_score = score
                    best_feature = j
                    best_threshold = t
        return best_feature, best_threshold

    def random_split(self):
        best_score = 10000000
        best_feature = None
        best_threshold = None
        for j in np.random.choice(self.X.shape[1], int(np.sqrt(self.X.shape[1])), replace=False):
            t = np.random.uniform(self.X[:, j].min(), self.X[:, j].max())
            N = len(self.y)
            left_mask = self.X[:, j] <= t
            left_node = NodeClassifier(self.X[left_mask], self.y[left_mask])
            right_node = NodeClassifier(self.X[~left_mask], self.y[~left_mask])
            if len(left_node.y) == 0 or len(right_node.y) == 0: continue
            score = (len(left_node.y) / N) * left_node.gini() + (len(right_node.y) / N) * right_node.gini()
            if score < best_score:
                best_score = score
                best_feature = j
                best_threshold = t
        return best_feature, best_threshold

In [13]:
class myDecisionTreeClassifier:
    def __init__(self, max_depth, min_samples_split):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.root = NodeClassifier(np.array(X), np.array(y))
        self.build_tree(self.root, depth=0)

    def build_tree(self, node, depth):
        if depth >= self.max_depth or len(node.y) < self.min_samples_split or node.gini() == 0:
            node.value = node.classes[np.argmax(node.counts)]
        else:
            feature, threshold = node.split()
            if feature is None:
                node.value = np.mean(node.y)
                return
            node.feature = feature
            node.threshold = threshold
            left_mask = node.X[:, node.feature] <= node.threshold
            node.left = NodeClassifier(node.X[left_mask], node.y[left_mask])
            node.right = NodeClassifier(node.X[~left_mask], node.y[~left_mask])
            self.build_tree(node.left, depth + 1)
            self.build_tree(node.right, depth + 1)

    def check(self, x, node):
        if node.value != None:
            return node.value
        else:
            if x[node.feature] <= node.threshold:
                return self.check(x, node.left)
            else:
                return self.check(x, node.right)

    def check_proba(self, x, node):
        if node.value != None:
            proba = np.zeros(len(self.classes))
            idx = np.searchsorted(self.classes, node.classes)
            proba[idx] = node.counts / len(node.y)
            return proba
            #return node.counts / len(node.y)
        else:
            if x[node.feature] <= node.threshold:
                return self.check_proba(x, node.left)
            else:
                return self.check_proba(x, node.right)

    def predict(self, X):
        predictions = []
        for i in range(len(X)):
            predictions.append(self.check(X[i], self.root))
        return predictions

    def predict_proba(self, X):
        probas = []
        for i in range(len(X)):
            probas.append(self.check_proba(X[i], self.root))
        return np.array(probas)

In [14]:
class myExtraRandomizedTreeClassifier:
    def __init__(self, max_depth, min_samples_split):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.root = NodeClassifier(X, y)
        self.build_tree(self.root, depth=0)

    def build_tree(self, node, depth):
        if depth >= self.max_depth or len(node.y) < self.min_samples_split or node.gini() == 0:
            node.value = node.classes[np.argmax(node.counts)]
        else:
            node.feature, node.threshold = node.random_split()
            left_mask = node.X[:, node.feature] <= node.threshold
            node.left = NodeClassifier(node.X[left_mask], node.y[left_mask])
            node.right = NodeClassifier(node.X[~left_mask], node.y[~left_mask])
            self.build_tree(node.left, depth + 1)
            self.build_tree(node.right, depth + 1)

    def check(self, x, node):
        if node.value != None:
            return node.value
        else:
            if x[node.feature] <= node.threshold:
                return self.check(x, node.left)
            else:
                return self.check(x, node.right)

    def check_proba(self, x, node):
        if node.value != None:
            proba = np.zeros(len(self.classes))
            idx = np.searchsorted(self.classes, node.classes)
            proba[idx] = node.counts / len(node.y)
            return proba
        else:
            if x[node.feature] <= node.threshold:
                return self.check_proba(x, node.left)
            else:
                return self.check_proba(x, node.right)

    def predict(self, X):
        predictions = []
        for i in range(len(X)):
            predictions.append(self.check(X[i], self.root))
        return predictions

    def predict_proba(self, X):
        probas = []
        for i in range(len(X)):
            probas.append(self.check_proba(X[i], self.root))
        return np.array(probas)

In [15]:
class NodeRegressor:
    def __init__(self, X, y, max_features):
        self.X = X
        self.y = y
        self.max_features = max_features
        self.feature = None
        self.left = None
        self.right = None
        self.threshold = None
        self.value = None

    def std(self):
        return np.std(self.y)

    def split(self):
        best_score = 10000000
        best_feature = None
        best_threshold = None
        features = np.random.choice(self.X.shape[1], self.max_features, replace=False)
        for j in features:
            for t in np.unique(self.X[:, j]):
                N = len(self.y)
                left_mask = self.X[:, j] <= t
                yl = self.y[left_mask]
                yr = self.y[~left_mask]
                if len(yl) == 0 or len(yr) == 0: continue
                score = (len(yl) / N) * np.std(yl) + (len(yr) / N) * np.std(yr)
                if score < best_score:
                    best_score = score
                    best_feature = j
                    best_threshold = t
        return best_feature, best_threshold

    def random_split(self):
        best_score = 10000000
        best_feature = None
        best_threshold = None
        for j in np.random.choice(self.X.shape[1], int(np.sqrt(self.X.shape[1])), replace=False):
            t = np.random.uniform(self.X[:, j].min(), self.X[:, j].max())
            N = len(self.y)
            left_mask = self.X[:, j] <= t
            left_node = NodeRegressor(self.X[left_mask], self.y[left_mask], max_features=self.max_features)
            right_node = NodeRegressor(self.X[~left_mask], self.y[~left_mask], max_features=self.max_features)
            if len(left_node.y) == 0 or len(right_node.y) == 0: continue
            score = (len(left_node.y) / N) * left_node.std() + (len(right_node.y) / N) * right_node.std()
            if score < best_score:
                best_score = score
                best_feature = j
                best_threshold = t
        return best_feature, best_threshold

In [16]:
class myDecisionTreeRegressor:
    def __init__(self, max_depth, min_samples_split, max_features):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None

    def fit(self, X, y):
        self.root = NodeRegressor(X, y, max_features=self.max_features)
        self.build_tree(self.root, depth=0)

    def build_tree(self, node, depth):
        if depth >= self.max_depth or len(node.y) < self.min_samples_split or node.std() == 0:
            node.value = np.mean(node.y)
        else:
            node.feature, node.threshold = node.split()
            left_mask = node.X[:, node.feature] <= node.threshold
            node.left = NodeRegressor(node.X[left_mask], node.y[left_mask], self.max_features)
            node.right = NodeRegressor(node.X[~left_mask], node.y[~left_mask], self.max_features)
            self.build_tree(node.left, depth + 1)
            self.build_tree(node.right, depth + 1)

    def check(self, x, node):
        if node.value != None:
            return node.value
        else:
            if x[node.feature] <= node.threshold:
                return self.check(x, node.left)
            else:
                return self.check(x, node.right)

    def predict(self, X):
        predictions = []
        for i in range(len(X)):
            predictions.append(self.check(X[i], self.root))
        return np.array(predictions)

In [17]:
class myExtraRandomizedTreeRegressor:
    def __init__(self, max_depth, min_samples_split):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = NodeRegressor(X, y)
        self.build_tree(self.root, depth=0)

    def build_tree(self, node, depth):
        if depth >= self.max_depth or len(node.y) < self.min_samples_split or node.std() == 0:
            node.value = np.mean(node.y)
        else:
            node.feature, node.threshold = node.random_split()
            left_mask = node.X[:, node.feature] <= node.threshold
            node.left = NodeRegressor(node.X[left_mask], node.y[left_mask])
            node.right = NodeRegressor(node.X[~left_mask], node.y[~left_mask])
            self.build_tree(node.left, depth + 1)
            self.build_tree(node.right, depth + 1)

    def check(self, x, node):
        if node.value != None:
            return node.value
        else:
            if x[node.feature] <= node.threshold:
                return self.check(x, node.left)
            else:
                return self.check(x, node.right)

    def predict(self, X):
        predictions = []
        for i in range(len(X)):
            predictions.append(self.check(X[i], self.root))
        return predictions

### 3. With your DecisionTree module, you must obtain a Gini score of at least 0.1 on the validation dataset.

In [18]:
def my_roc_auc(y_class, y_pred):
    positive = (y_class == 1).sum()
    negative = (y_class == 0).sum()

    sorted_idx = np.argsort(y_pred)[::-1]
    y_class_sorted = y_class.iloc[sorted_idx]
    TP = 0
    FP = 0
    points = [(0, 0)]

    for klass in y_class_sorted:
        if klass == 1:
            TP += 1
        else:
            FP += 1
        TPR = TP / positive
        FPR = FP / negative
        points.append((FPR, TPR))

    auc = 0
    for i in range(len(points) - 1):
        x1, y1 = points[i]
        x2, y2 = points[i + 1]
        width = x2 - x1
        height = (y2 + y1) / 2
        auc += width * height

    return auc

In [19]:
def gini_metric(auc_score):
    gini_met = abs(2 * auc_score - 1)
    return gini_met

In [20]:
mytreeclass = myDecisionTreeClassifier(max_depth=5, min_samples_split=2)
mytreeclass.fit(X_train, y_train)
class_predict = mytreeclass.predict_proba(X_valid)
gini_metric(my_roc_auc(y_valid, class_predict[:, 1]))

np.float64(0.2957387911847904)

### 4. Use sklearn's DecisionTreeClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?

In [41]:
sklearn_tree_class = DecisionTreeClassifier(max_depth=5, min_samples_split=2, random_state=21)
sklearn_tree_class.fit(X_train, y_train)
sklearn_class_predict = sklearn_tree_class.predict_proba(X_valid)
gini_metric(my_roc_auc(y_valid, sklearn_class_predict[:, 1]))

np.float64(0.2799055269255284)

mine is even better xD LOL

### 5. Implement the RandomForestClassifier and check its performance. You have to improve the result of a single tree and get at least 0.15 Gini score on the validation dataset. Be able to set a fixed random seed.

In [22]:
class myRandomForestClassifier():
    def __init__(self, n_estimators, max_depth, min_samples_split, random_state=21):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        np.random.seed(self.random_state)
        X = np.array(X)
        y = np.array(y)
        N = len(y)
        for tree in range(self.n_estimators):
            idx = np.random.choice(N, N, replace=True)
            X_sample = X[idx]
            y_sample = y[idx]
            fitted_tree = myExtraRandomizedTreeClassifier(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            fitted_tree.fit(X_sample, y_sample)
            self.trees.append(fitted_tree)

    def predict_proba(self, X):
        probas = []
        for tree in self.trees:
            proba = tree.predict_proba(X)
            probas.append(proba)
        return np.mean(probas, axis=0)

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

In [23]:
myforest = myRandomForestClassifier(max_depth=5, min_samples_split=2, n_estimators=50)
myforest.fit(X_train, y_train)
probas = myforest.predict_proba(X_valid)[:, 1]
gini_metric(my_roc_auc(y_valid, probas))

np.float64(0.3229766598735513)

### 6. Use your DecisionTree design class for GBDT classifier. This class must have max_depth, number_of_trees and max_features attributes. You must compute the gradient of the binary cross-entropy loss function and implement incremental learning: train the next tree using the results of the previous trees.

In [24]:
class myGBDT:
    def __init__(self, n_estimators, max_depth, min_samples_split, learning_rate, max_features):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.learning_rate = learning_rate
        self.max_features = max_features
        self.F0 = None
        self.trees = []

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def pseudepodiy(self, y, F_pred):
        r = y - self.sigmoid(F_pred)
        return r

    def F_predict(self, X):
        F = np.full(len(X), self.F0)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        p0 = np.mean(y)
        self.F0 = np.log(p0 / (1 - p0))
        F_pred = np.full(len(y), self.F0)
        for i in range(self.n_estimators):
            r = self.pseudepodiy(y, F_pred)
            tree = myDecisionTreeRegressor(max_depth=self.max_depth, min_samples_split=self.min_samples_split, max_features=self.max_features)
            tree.fit(X, r)
            self.trees.append(tree)
            F_pred += self.learning_rate * tree.predict(X)

    def predict_proba(self, X):
        F = self.F_predict(X)
        p = self.sigmoid(F)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)


In [25]:
boosting = myGBDT(n_estimators=10, max_depth=5, min_samples_split=2, learning_rate=0.1, max_features=10)
boosting.fit(X_train, y_train)
probas = boosting.predict_proba(X_valid)[:, 1]
gini_metric(my_roc_auc(y_valid, probas))

np.float64(0.34101204699323184)

### 7. Use LightGBM, Catboost, and XGBoost for fitting on a training set and prediction on a validation set. Review the documentation of the libraries and fine-tune the algorithms for the task. Note key differences between each implementation. Analyze special features of each algorithm (how does "categorical feature" work in Catboost, what is DART mode in XGBoost)? Which GBDT model gives the best result? Can you explain why?

In [31]:
cat_model = CatBoostClassifier(iterations=1000, learning_rate=0.1, eval_metric='NormalizedGini',
                               loss_function='Logloss', random_seed=21)
cat_model.fit(X_train, y_train, eval_set=(X_valid, y_valid))
cat_probas = cat_model.predict_proba(X_valid)[:, 1]
gini_metric(my_roc_auc(y_valid, cat_probas))


0:	test: 0.0913254	best: 0.0913254 (0)	total: 21.9ms	remaining: 21.9s
1:	test: 0.1865500	best: 0.1865500 (1)	total: 45.8ms	remaining: 22.8s
2:	test: 0.2089727	best: 0.2089727 (2)	total: 67.8ms	remaining: 22.5s
3:	test: 0.2603447	best: 0.2603447 (3)	total: 87.8ms	remaining: 21.9s
4:	test: 0.3108070	best: 0.3108070 (4)	total: 109ms	remaining: 21.7s
5:	test: 0.3022806	best: 0.3108070 (4)	total: 129ms	remaining: 21.4s
6:	test: 0.3097111	best: 0.3108070 (4)	total: 150ms	remaining: 21.3s
7:	test: 0.3116605	best: 0.3116605 (7)	total: 171ms	remaining: 21.2s
8:	test: 0.3159573	best: 0.3159573 (8)	total: 191ms	remaining: 21s
9:	test: 0.3237349	best: 0.3237349 (9)	total: 216ms	remaining: 21.4s
10:	test: 0.3251312	best: 0.3251312 (10)	total: 240ms	remaining: 21.6s
11:	test: 0.3263969	best: 0.3263969 (11)	total: 262ms	remaining: 21.5s
12:	test: 0.3258859	best: 0.3263969 (11)	total: 281ms	remaining: 21.4s
13:	test: 0.3271464	best: 0.3271464 (13)	total: 301ms	remaining: 21.2s
14:	test: 0.3319647	best

np.float64(0.3518365483737165)

In [35]:
lgb_model = LGBMClassifier(n_estimators=50, learning_rate=0.1, metric='auc', objective='binary',
                           random_state=21)
lgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])
lgb_probas = lgb_model.predict_proba(X_valid)[:, 1]
gini_metric(my_roc_auc(y_valid, lgb_probas))

[LightGBM] [Info] Number of positive: 2840, number of negative: 21855
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003321 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3330
[LightGBM] [Info] Number of data points in the train set: 24695, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.115003 -> initscore=-2.040626
[LightGBM] [Info] Start training from score -2.040626


C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.3624607647404261)

In [38]:
xgd_model = XGBClassifier(n_estimators=50, max_depth=5, min_samples_split=2, random_state=21, eval_metric='auc', objective='binary:logistic', learning_rate=0.1)
xgd_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])
xgd_probas = xgd_model.predict_proba(X_valid)[:, 1]
gini_metric(my_roc_auc(y_valid, xgd_probas))

[0]	validation_0-auc:0.65363
[1]	validation_0-auc:0.66688
[2]	validation_0-auc:0.66805
[3]	validation_0-auc:0.67120
[4]	validation_0-auc:0.67555
[5]	validation_0-auc:0.67415
[6]	validation_0-auc:0.67456
[7]	validation_0-auc:0.67470
[8]	validation_0-auc:0.67422
[9]	validation_0-auc:0.67513
[10]	validation_0-auc:0.67570
[11]	validation_0-auc:0.67587
[12]	validation_0-auc:0.67811


C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:31:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[13]	validation_0-auc:0.67822
[14]	validation_0-auc:0.67951
[15]	validation_0-auc:0.68072
[16]	validation_0-auc:0.68136
[17]	validation_0-auc:0.68204
[18]	validation_0-auc:0.68119
[19]	validation_0-auc:0.68156
[20]	validation_0-auc:0.68240
[21]	validation_0-auc:0.68386
[22]	validation_0-auc:0.68362
[23]	validation_0-auc:0.68374
[24]	validation_0-auc:0.68290
[25]	validation_0-auc:0.68292
[26]	validation_0-auc:0.68303
[27]	validation_0-auc:0.68431
[28]	validation_0-auc:0.68481
[29]	validation_0-auc:0.68499
[30]	validation_0-auc:0.68425
[31]	validation_0-auc:0.68441
[32]	validation_0-auc:0.68474
[33]	validation_0-auc:0.68453
[34]	validation_0-auc:0.68552
[35]	validation_0-auc:0.68624
[36]	validation_0-auc:0.68674
[37]	validation_0-auc:0.68621
[38]	validation_0-auc:0.68618
[39]	validation_0-auc:0.68629
[40]	validation_0-auc:0.68667
[41]	validation_0-auc:0.68687
[42]	validation_0-auc:0.68675
[43]	validation_0-auc:0.68689
[44]	validation_0-auc:0.68685
[45]	validation_0-auc:0.68658
[46]	valid

np.float64(0.37264987404702454)

lightgbm среди всех листьев ищет тот, который принесет максимальное уменьшение ошибки и делит только его. дерево растет вглубь ассиметрично

быстрый, но склонен к переобучению

xgboost наоборот, строит деревья горизонтально, разделяет все узлы на уровне, даже если какой то из них принесет мало пользы

строит сбалансированные и стабильные деревья, но требует много ресурсов

catboost также строит симметричные деревья, но на одном уровне для всех узлов используется один и тот же признак и сплит

строится моментально, сама структура является регуляризатором, но требует большого количества деревьев для сложных зависимостей

при работе с категориальными признаками лучше всего себя показывает catboost

catboost с категориальными признаками работает так:
1 случайно перемешивает строки
2 далее работает как таргет енкодинг, но берет только прошлые значения, а следующие не берет
так же используются комбинации признаков

DART в xgboost нужен для избежания переобучения и затухания важности деревьев, идущих после первых 30-50 деревьев.
он просто выбрасывает некоторые деревья, чтобы ошибка считалась только на оставшихся, таким образом новое дерево больше влияет на ошибку.

best model is xgboost

### 8. Take the best model and estimate its performance on the test dataset: check the Gini values on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitting or not? Explain.

In [39]:
xgd_model = XGBClassifier(n_estimators=50, max_depth=5, min_samples_split=2, random_state=21, eval_metric='auc', objective='binary:logistic', learning_rate=0.1)
xgd_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])
xgd_probas = xgd_model.predict_proba(X_valid)[:, 1]
gini_metric(my_roc_auc(y_valid, xgd_probas))

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:57:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-auc:0.65363
[1]	validation_0-auc:0.66688
[2]	validation_0-auc:0.66805
[3]	validation_0-auc:0.67120
[4]	validation_0-auc:0.67555
[5]	validation_0-auc:0.67415
[6]	validation_0-auc:0.67456
[7]	validation_0-auc:0.67470
[8]	validation_0-auc:0.67422
[9]	validation_0-auc:0.67513
[10]	validation_0-auc:0.67570
[11]	validation_0-auc:0.67587
[12]	validation_0-auc:0.67811
[13]	validation_0-auc:0.67822
[14]	validation_0-auc:0.67951
[15]	validation_0-auc:0.68072
[16]	validation_0-auc:0.68136
[17]	validation_0-auc:0.68204
[18]	validation_0-auc:0.68119
[19]	validation_0-auc:0.68156
[20]	validation_0-auc:0.68240
[21]	validation_0-auc:0.68386
[22]	validation_0-auc:0.68362
[23]	validation_0-auc:0.68374
[24]	validation_0-auc:0.68290
[25]	validation_0-auc:0.68292
[26]	validation_0-auc:0.68303
[27]	validation_0-auc:0.68431
[28]	validation_0-auc:0.68481
[29]	validation_0-auc:0.68499
[30]	validation_0-auc:0.68425
[31]	validation_0-auc:0.68441
[32]	validation_0-auc:0.68474
[33]	validation_0-au

np.float64(0.37264987404702454)

In [40]:
xgd_model = XGBClassifier(n_estimators=50, max_depth=5, min_samples_split=2, random_state=21, eval_metric='auc', objective='binary:logistic', learning_rate=0.1)
xgd_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
xgd_probas = xgd_model.predict_proba(X_test)[:, 1]
gini_metric(my_roc_auc(y_test, xgd_probas))

[0]	validation_0-auc:0.66529
[1]	validation_0-auc:0.66860
[2]	validation_0-auc:0.67001
[3]	validation_0-auc:0.67412
[4]	validation_0-auc:0.68154
[5]	validation_0-auc:0.67872
[6]	validation_0-auc:0.68096
[7]	validation_0-auc:0.68111
[8]	validation_0-auc:0.68134
[9]	validation_0-auc:0.68243
[10]	validation_0-auc:0.68275
[11]	validation_0-auc:0.68250
[12]	validation_0-auc:0.68315
[13]	validation_0-auc:0.68344
[14]	validation_0-auc:0.68441
[15]	validation_0-auc:0.68606
[16]	validation_0-auc:0.68663
[17]	validation_0-auc:0.68767
[18]	validation_0-auc:0.68733
[19]	validation_0-auc:0.68743
[20]	validation_0-auc:0.68803


C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:57:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[21]	validation_0-auc:0.68831
[22]	validation_0-auc:0.68762
[23]	validation_0-auc:0.68837
[24]	validation_0-auc:0.68774
[25]	validation_0-auc:0.68538
[26]	validation_0-auc:0.68408
[27]	validation_0-auc:0.68430
[28]	validation_0-auc:0.68436
[29]	validation_0-auc:0.68430
[30]	validation_0-auc:0.68373
[31]	validation_0-auc:0.68380
[32]	validation_0-auc:0.68392
[33]	validation_0-auc:0.68414
[34]	validation_0-auc:0.68375
[35]	validation_0-auc:0.68419
[36]	validation_0-auc:0.68383
[37]	validation_0-auc:0.68341
[38]	validation_0-auc:0.68331
[39]	validation_0-auc:0.68257
[40]	validation_0-auc:0.68252
[41]	validation_0-auc:0.68272
[42]	validation_0-auc:0.68275
[43]	validation_0-auc:0.68286
[44]	validation_0-auc:0.68261
[45]	validation_0-auc:0.68227
[46]	validation_0-auc:0.68242
[47]	validation_0-auc:0.68250
[48]	validation_0-auc:0.68218
[49]	validation_0-auc:0.68202


np.float64(0.3640474204640429)

i dont think that my model is overfitted, cuz there is no huge gini drop